In [1]:
from pathlib import Path
import numpy as np
import SimpleITK as sitk
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display

IMAGES_TR = Path(
    "/cluster/projects/vc/data/mic/open/Prostate/PI-CAI-V2.0"
    "/workdir/nnUNet_raw_data/Task2203_picai_baseline/imagesTr"
)
ZONES_BASE = Path(
    "/cluster/projects/vc/data/mic/open/Prostate/PI-CAI-V2.0"
    "/picai_labels/anatomical_delineations/zonal_pz_tz/AI"
)

cases = sorted({p.name.replace('_0000.nii.gz', '')
                for p in IMAGES_TR.glob('*_0000.nii.gz')})
print(f"{len(cases)} cases found")

1499 cases found


In [2]:
_img_cache: dict = {}
_zone_cache: dict = {}

def load_image(case_id: str) -> np.ndarray:
    """Return (3, D, H, W) float32 array: [T2W, ADC, HBV]."""
    if case_id in _img_cache:
        return _img_cache[case_id]
    imgs = []
    for ch in ('0000', '0001', '0002'):
        itk = sitk.ReadImage(str(IMAGES_TR / f'{case_id}_{ch}.nii.gz'))
        imgs.append(sitk.GetArrayFromImage(itk).astype(np.float32))
    result = np.stack(imgs)  # (3, D, H, W)
    _img_cache[case_id] = result
    return result

def load_zone(case_id: str) -> np.ndarray | None:
    """Return (D, H, W) int8 zone map resampled to imagesTr space, or None."""
    if case_id in _zone_cache:
        return _zone_cache[case_id]
    zone_path = None
    for ver in ('Yuan23', 'HeviAI23'):
        p = ZONES_BASE / ver / f'{case_id}.nii.gz'
        if p.exists():
            zone_path = p
            break
    if zone_path is None:
        _zone_cache[case_id] = None
        return None
    ref_itk  = sitk.ReadImage(str(IMAGES_TR / f'{case_id}_0000.nii.gz'))
    zone_itk = sitk.ReadImage(str(zone_path))
    rs = sitk.ResampleImageFilter()
    rs.SetReferenceImage(ref_itk)
    rs.SetInterpolator(sitk.sitkNearestNeighbor)
    rs.SetDefaultPixelValue(0)
    result = sitk.GetArrayFromImage(rs.Execute(zone_itk)).astype(np.int8)
    _zone_cache[case_id] = result
    return result

In [3]:
CH_NAMES = ['T2W', 'ADC', 'HBV']
TZ_COLOR = (0.15, 0.35, 1.00, 0.45)
PZ_COLOR = (0.10, 0.85, 0.20, 0.45)

w_case    = widgets.Dropdown(options=cases, description='Case:')
w_slice   = widgets.IntSlider(min=0, max=24, value=12, description='Slice:')
w_channel = widgets.RadioButtons(options=CH_NAMES, value='T2W', description='Channel:')
w_zones   = widgets.Checkbox(value=True, description='Show zones')

out = widgets.Output()

def update_slice_range(case_id):
    img = load_image(case_id)
    D = img.shape[1]
    w_slice.max   = D - 1
    w_slice.value = D // 2

def on_case_change(change):
    update_slice_range(change['new'])

w_case.observe(on_case_change, names='value')
update_slice_range(w_case.value)

def show(case_id, slice_idx, channel, show_zones):
    img  = load_image(case_id)
    zone = load_zone(case_id) if show_zones else None

    ch_idx = CH_NAMES.index(channel)
    sl = img[ch_idx, slice_idx]          # (H, W)

    # Robust normalisation
    lo, hi = np.percentile(sl, (1, 99))
    sl_norm = np.clip((sl - lo) / (hi - lo + 1e-8), 0, 1)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(sl_norm, cmap='gray', aspect='equal', interpolation='nearest')

    legend_patches = []
    if zone is not None and slice_idx < zone.shape[0]:
        zone_sl = zone[slice_idx]        # (H, W)
        rgba = np.zeros((*zone_sl.shape, 4), dtype=np.float32)
        rgba[zone_sl == 1] = TZ_COLOR
        rgba[zone_sl == 2] = PZ_COLOR
        ax.imshow(rgba, aspect='equal', interpolation='nearest')
        legend_patches = [
            mpatches.Patch(facecolor=TZ_COLOR[:3] + (0.7,), label='TZ'),
            mpatches.Patch(facecolor=PZ_COLOR[:3] + (0.7,), label='PZ'),
        ]

    ax.set_title(f'{case_id}  |  {channel}  |  slice {slice_idx}')
    ax.axis('off')
    if legend_patches:
        ax.legend(handles=legend_patches, loc='lower right', framealpha=0.7)
    plt.tight_layout()
    plt.show()

ui = widgets.VBox([
    widgets.HBox([w_case, w_channel, w_zones]),
    w_slice,
])
out = widgets.interactive_output(
    show,
    dict(case_id=w_case, slice_idx=w_slice, channel=w_channel, show_zones=w_zones),
)
display(ui, out)

Output()